In [3]:
import numpy as np
import torch
import os

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
from transformers import AutoModel
from sklearn.linear_model import LogisticRegression

import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd



# GPU oder CPU auswählen

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU wird benutzt.")
else:
    device = torch.device("cpu")
    print("CPU wird benutzt.")

print("Device:", device)

jubaea_directory = "../Data/Jubaea"
not_jubaea_directory = "../Data/Not_Jubaea"
img_format = (".jpg", ".jpeg", ".png", ".webp")

#Erstelle Klassennamen aus den Ordnern
class_names = ["Jubaea Chilensis"] +sorted(
    d for d in os.listdir(not_jubaea_directory)
    if os.path.isdir(os.path.join(not_jubaea_directory,d))    #Übergibt nur Namen der Ordner,nicht der Datein
)

#Klassennamen Zahlen zuweisen
class_enumerate = {name :i for i, name in enumerate(class_names)}
class_id = len(class_enumerate)


#  Daten importieren
def create_sample():
    sample = []

    for image in os.listdir(jubaea_directory):
        if image.lower().endswith((img_format)):
            sample.append(
                (os.path.join(jubaea_directory, image), class_enumerate["Jubaea Chilensis"])
            )


    #Iteriere für alle anderen Palmenarten durch
    for name in class_names:

        folder_path = os.path.join(
            not_jubaea_directory,
            name
        )

        if not os.path.isdir(folder_path):
            continue

        for image in os.listdir(folder_path):

            if image.lower().endswith((img_format)):
                sample.append((os.path.join(folder_path,image),class_enumerate[name]))

    return sample


sample = create_sample()

pictures = [x[0] for x in sample]
labels = [x[1] for x in sample]

#Daten cappen
data_fraction = 1       #Anpassen
if data_fraction < 1:
    pictures,_, labels,_ = train_test_split(
        pictures,
        labels,
        train_size=data_fraction,
        stratify=labels,
        random_state=42
    )


#Neuer Trainingsplit mit Validation 70/15/15

#Hier 70% Data zum trainieren
X_train, X_temp, Y_train, Y_temp = train_test_split(
    pictures,
    labels,
    test_size=0.3,
    stratify=labels,
    random_state=42
)

# 15/15 Val und Testing
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.5,
    stratify=Y_temp,
    random_state=42
)

print("Training: ",len(X_train))
print("Validation: ",len(X_val))
print("Testing: ",len(X_test))


# Transformationen


# Werte, die beim ursprünglichen Training von ResNet18 auf ImageNet benutzt wurden
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]


# Resize + Data Augmentation + Normalisierung
train_transform = transforms.Compose([

    transforms.RandomResizedCrop(224,scale = (0.8,1.0)),

    transforms.RandomRotation(15),

    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])


test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])



# Dataset


class JubaeaDataset(Dataset):

    def __init__(
        self,
        paths,
        labels,
        transform=None
    ):

        self.paths = paths
        self.labels = labels
        self.transform = transform


    def __len__(self):
        return len(self.paths)


    def __getitem__(self, index):

        path = self.paths[index]

        label = self.labels[index]

        image = Image.open(path).convert("RGB")

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = JubaeaDataset(
    X_train,
    Y_train,
    transform=train_transform
)

val_dataset = JubaeaDataset(
    X_val,
    Y_val,
    transform=test_transform
)

test_dataset = JubaeaDataset(
    X_test,
    Y_test,
    transform=test_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)


# DINOv2

'''
backbone = AutoModel.from_pretrained("facebook/dinov2-base").to(device).eval()
@torch.no_grad()
def extract(loader):
    feats, ys  =[],[]
    for images, labels in loader:
        h = backbone(pixel_values=images.to(device)).last_hidden_state
        f = torch.cat([h[:,0], h[:,1:].mean(1)], dim=1)
        feats.append(f.cpu()); ys.append(labels)
    return torch.cat(feats).numpy(), torch.cat(ys).numpy()

Xtr, Ytr = extract(train_loader)
Xva, Yva = extract(val_loader)

clf = LogisticRegression(max_iter=10000, C=1.0)
clf.fit(Xtr, Ytr)
print("Val Acc:" , clf.score(Xva, Yva))

'''

class DinoClassifier(nn.Module):
    def __init__(self, n_classes, freeze_layers=6):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov2-base")
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(768 * 2, n_classes))
        for p in self.backbone.embeddings.parameters():
            p.requires_grad = False
        for layer in self.backbone.encoder.layer[:freeze_layers]:
            for p in layer.parameters():
                p.requires_grad = False

    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.head(torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1))

model = DinoClassifier(class_id).to(device)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

epochs = 8
optimizer = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 1e-5},
    {"params": model.head.parameters(), "lr": 1e-3},
], weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[1e-5, 1e-3],
    total_steps=epochs * len(train_loader), pct_start=0.1
)

best_val_acc = 0
for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = loss_fn(out, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    model.eval()
    vc, vt = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            out = model(images.to(device))
            vc += (out.argmax(1).cpu() == labels).sum().item()
            vt += labels.size(0)
    val_acc = vc / vt
    print(f"Epoch {epoch+1}/{epochs} | loss {total_loss/len(train_loader):.3f} | "
          f"train {correct/total:.3f} | val {val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "../dinov2_ver1.0.pth")



GPU wird benutzt.
Device: mps
Training:  5971
Validation:  1280
Testing:  1280


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 7652.21it/s]


Epoch 1/8 | loss 1.507 | train 0.577 | val 0.795
Epoch 2/8 | loss 0.941 | train 0.828 | val 0.853
Epoch 3/8 | loss 0.826 | train 0.879 | val 0.845
Epoch 4/8 | loss 0.756 | train 0.906 | val 0.853
Epoch 5/8 | loss 0.699 | train 0.932 | val 0.855
Epoch 6/8 | loss 0.671 | train 0.945 | val 0.848


KeyboardInterrupt: 

In [4]:
model.load_state_dict(torch.load("../dinov2_ver1.0.pth", map_location=device))
model.eval()
preds, ys = [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds += model(images.to(device)).argmax(1).cpu().tolist()
        ys += labels.tolist()
print(classification_report(ys, preds, target_names=class_names))

                       precision    recall  f1-score   support

     Jubaea Chilensis       0.96      0.95      0.96       147
   Adonidia merrillii       0.96      0.99      0.98       105
        Butia Odarata       0.91      0.90      0.90       108
       Cocos Nucifera       0.93      0.93      0.93       105
  Phoenix Canariensis       0.61      0.81      0.70       114
  Phoenix Dactylifera       0.77      0.57      0.66       131
      Roystonea Regia       0.90      0.92      0.91       113
Syagrus romanzoffiana       0.90      0.83      0.86       105
Trachycarpus Fortunei       0.94      0.94      0.94       122
Washingtonia filifera       0.70      0.82      0.75       118
 Washingtonia_Robusta       0.78      0.65      0.71       112

             accuracy                           0.85      1280
            macro avg       0.85      0.85      0.85      1280
         weighted avg       0.85      0.85      0.84      1280

